# Comprehensive Capstone Project
### Preprocessing, Supervised Machine Learning, Ensembles, Tuning, and LLM explanation Pipeline

This notebook contains all executable steps for **Part 1**, **Part 2**, **Part 3**, and **Part 4**.

## 1. Setup & Requirements

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import joblib
from jsonschema import validate, ValidationError

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    mean_squared_error, r2_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score, precision_score, recall_score, f1_score
)
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
%matplotlib inline
print("All libraries loaded successfully.")

## 2. Load Cleaned Dataset

In [ ]:
df = pd.read_csv('cleaned_data.csv')
print(f"Cleaned dataset loaded. Shape: {df.shape[0]} rows × {df.shape[1]} columns.")

## 3. Supervised ML: Regression & Classification Preprocessing (Part 2)

In [ ]:
# Features and labels
X = df.drop(columns=['SalePrice', 'Id', 'Alley'], errors='ignore')
y_reg = df['SalePrice']
y_clf = (y_reg > y_reg.median()).astype(int)

# Preprocess categorical features via One-Hot encoding
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True, dtype=int)

# Split datasets
X_train, X_test, y_reg_train, y_reg_test = train_test_split(X_encoded, y_reg, test_size=0.2, random_state=42)
_, _, y_clf_train, y_clf_test = train_test_split(X_encoded, y_clf, test_size=0.2, random_state=42)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

## 4. Part 2 — Regression Evaluation

In [ ]:
# OLS Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_reg_train)
y_pred_lr = lr.predict(X_test_scaled)
print(f"OLS R2: {r2_score(y_reg_test, y_pred_lr):.4f}")
print(f"OLS MSE: {mean_squared_error(y_reg_test, y_pred_lr):.2f}")

# Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_reg_train)
y_pred_ridge = ridge.predict(X_test_scaled)
print(f"Ridge R2: {r2_score(y_reg_test, y_pred_ridge):.4f}")
print(f"Ridge MSE: {mean_squared_error(y_reg_test, y_pred_ridge):.2f}")

# Top 3 OLS coefficients
coef_summary = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': lr.coef_})
coef_summary['Abs_Coef'] = coef_summary['Coefficient'].abs()
display(coef_summary.sort_values('Abs_Coef', ascending=False).head(3))

## 5. Part 2 — Classification (Logistic Regression & Bootstrap)

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_clf_train)
y_pred_clf = log_reg.predict(X_test_scaled)
y_prob_clf = log_reg.predict_proba(X_test_scaled)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_clf_test, y_pred_clf))
print("\nClassification Report:")
print(classification_report(y_clf_test, y_pred_clf))
print(f"ROC-AUC: {roc_auc_score(y_clf_test, y_prob_clf):.4f}")

### Plot ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_clf_test, y_prob_clf)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Baseline C=1.0 (AUC = {roc_auc_score(y_clf_test, y_prob_clf):.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression')
plt.legend(loc="lower right")
plt.show()

### Threshold Sensitivity Table

In [ ]:
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
records = []
for t in thresholds:
    y_pred_t = (y_prob_clf >= t).astype(int)
    p = precision_score(y_clf_test, y_pred_t)
    r = recall_score(y_clf_test, y_pred_t)
    f1 = f1_score(y_clf_test, y_pred_t)
    records.append({'Threshold': t, 'Precision': p, 'Recall': r, 'F1': f1})
display(pd.DataFrame(records))

### Regularization Experiment & Bootstrap

In [ ]:
log_reg_strong = LogisticRegression(C=0.01, max_iter=1000, random_state=42)
log_reg_strong.fit(X_train_scaled, y_clf_train)
y_prob_clf_strong = log_reg_strong.predict_proba(X_test_scaled)[:, 1]

print(f"C=0.01 model ROC-AUC: {roc_auc_score(y_clf_test, y_prob_clf_strong):.4f}")

np.random.seed(42)
bootstrapped_diffs = []
y_clf_test_arr = np.array(y_clf_test)
for _ in range(500):
    indices = np.random.choice(len(y_clf_test_arr), size=len(y_clf_test_arr), replace=True)
    auc_1 = roc_auc_score(y_clf_test_arr[indices], y_prob_clf[indices])
    auc_01 = roc_auc_score(y_clf_test_arr[indices], y_prob_clf_strong[indices])
    bootstrapped_diffs.append(auc_1 - auc_01)

print(f"Mean AUC Diff: {np.mean(bootstrapped_diffs):.5f}")
print(f"95% Confidence Interval: [{np.percentile(bootstrapped_diffs, 2.5):.5f}, {np.percentile(bootstrapped_diffs, 97.5):.5f}]")

## 6. Part 3 — Advanced Modeling & Ensembles

In [ ]:
# Unconstrained Tree vs Controlled Tree
dt_un = DecisionTreeClassifier(random_state=42).fit(X_train_scaled, y_clf_train)
dt_co = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42).fit(X_train_scaled, y_clf_train)

print(f"Unconstrained Tree Train/Test Acc: {dt_un.score(X_train_scaled, y_clf_train):.4f} / {dt_un.score(X_test_scaled, y_clf_test):.4f}")
print(f"Controlled Tree Train/Test Acc: {dt_co.score(X_train_scaled, y_clf_train):.4f} / {dt_co.score(X_test_scaled, y_clf_test):.4f}")

# Gini vs Entropy
dt_gini = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42).fit(X_train_scaled, y_clf_train)
dt_entr = DecisionTreeClassifier(max_depth=5, criterion='entropy', random_state=42).fit(X_train_scaled, y_clf_train)
print(f"Gini Test Acc: {dt_gini.score(X_test_scaled, y_clf_test):.4f}, Entropy Test Acc: {dt_entr.score(X_test_scaled, y_clf_test):.4f}")

### Random Forest & Gradient Boosting

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42).fit(X_train_scaled, y_clf_train)
print(f"RF Test AUC: {roc_auc_score(y_clf_test, rf.predict_proba(X_test_scaled)[:, 1]):.4f}")

# Top 5 Features
rf_importances = pd.DataFrame({'Feature': X_train.columns, 'Importance': rf.feature_importances_})
display(rf_importances.sort_values('Importance', ascending=False).head(5))

# Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42).fit(X_train_scaled, y_clf_train)
print(f"GB Test AUC: {roc_auc_score(y_clf_test, gb.predict_proba(X_test_scaled)[:, 1]):.4f}")

### Feature Ablation Study

In [ ]:
# Identify 5 lowest importance features
lowest_5 = rf_importances.sort_values('Importance').head(5)['Feature'].tolist()
X_train_red = X_train.drop(columns=lowest_5)
X_test_red = X_test.drop(columns=lowest_5)

X_train_red_scaled = scaler.fit_transform(X_train_red)
X_test_red_scaled = scaler.transform(X_test_red)

rf_red = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42).fit(X_train_red_scaled, y_clf_train)
print(f"Full model test ROC-AUC: {roc_auc_score(y_clf_test, rf.predict_proba(X_test_scaled)[:, 1]):.5f}")
print(f"Reduced model test ROC-AUC: {roc_auc_score(y_clf_test, rf_red.predict_proba(X_test_red_scaled)[:, 1]):.5f}")

### Cross-Validated Comparison

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Controlled Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_clf_train, cv=cv_strategy, scoring='roc_auc')
    print(f"{name:25s} -> Mean AUC: {scores.mean():.4f} (Std: {scores.std():.4f})")

### Pipeline & GridSearchCV Tuning

In [ ]:
pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}

grid = GridSearchCV(pipeline, param_grid, cv=cv_strategy, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_clf_train)
print(f"Best params: {grid.best_params_}")
print(f"Best CV Score: {grid.best_score_:.4f}")

### Manual Learning Curve

In [ ]:
best_pipeline = grid.best_estimator_
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
records = []
for f in fractions:
    n_samples = int(f * len(X_train))
    X_sub = X_train.iloc[:n_samples]
    y_sub = y_clf_train.iloc[:n_samples]
    
    best_pipeline.fit(X_sub, y_sub)
    train_auc = roc_auc_score(y_sub, best_pipeline.predict_proba(X_sub)[:, 1])
    test_auc = roc_auc_score(y_clf_test, best_pipeline.predict_proba(X_test)[:, 1])
    records.append({'Fraction': f, 'Train AUC': train_auc, 'Test AUC': test_auc})

display(pd.DataFrame(records))

### Reload & Predict Block Verification

In [ ]:
# Serialize
joblib.dump(best_pipeline, 'best_model.pkl')
print("Model saved to best_model.pkl")

# Reload and run
loaded_model = joblib.load('best_model.pkl')
test_rows = X_test.iloc[:2].copy()
predictions = loaded_model.predict(test_rows)
print("Predictions:", predictions)
print("Probabilities:", loaded_model.predict_proba(test_rows)[:, 1])

## 7. Part 4 — LLM Prediction Explanation Pipeline

In [ ]:
# Execute LLM pipeline using the helper script to verify it
exec(open('part4_llm.py').read())